In [6]:
import warnings
import os
import json
from pathlib import Path
from glob import glob
from datetime import datetime

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
)
from sklearn.impute import SimpleImputer

import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm.notebook import tqdm

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("[INFO] pip install umap-learn for UMAP visualisation")

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", font_scale=1.1)
PALETTE = ["#4361EE", "#F72585", "#4CC9F0", "#7209B7",
           "#3A0CA3", "#560BAD", "#480CA8", "#B5179E"]
plt.rcParams.update({"figure.dpi": 120,
                     "axes.spines.top": False,
                     "axes.spines.right": False})

DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", "/Volumes/T5 EVO"))
DATASET_PATH = DATASET_ROOT / "hf" / "merrec"
assert DATASET_PATH.exists(), f"Not found: {DATASET_PATH}"
print(f"Ready | {datetime.now():%Y-%m-%d %H:%M:%S}")
print(f"Dataset: {DATASET_PATH}")

Ready | 2026-03-21 21:52:03
Dataset: /Volumes/T5 EVO/hf/merrec


In [11]:
CFG = {
    "col_user": "user_id",
    "col_item": "item_id",
    "col_action": "event_id",
    "col_time": "stime",
    "positive_action": "item_view",

    # 🔴 IMPORTANT: remove limit
    "max_total_rows": 10000,

    "session_gap_s": 1800,
    "min_session_events": 2,

    "features": [
        "session_length_s",
        "n_events",
        "n_unique_items",
        "revisit_rate",
        "inter_event_time_median",
        "session_velocity",
    ],

    "scaler": "robust",
    "log_transform_cols": [
        "session_length_s",
        "n_events",
        "n_unique_items",
        "inter_event_time_median",
    ],

    # 🔴 OPTIMIZED
    "k_range": range(2, 7),
    "covariance_types": ["full", "diag"],
    "n_init": 3,
    "max_iter": 300,

    "final_k": None,
    "final_cov_type": "full",

    # 🔴 REDUCED
    "bootstrap_n": 8,
    "bootstrap_sample_frac": 0.8,

    "output_dir": "gmm_outputs",
}

In [12]:
class MerRecSessionStreamer:
    """
    Streams Parquet files and yields one session-feature dict per
    completed session.  Mirrors MerRecStreamingCTR.__iter__ exactly.

    Fix: `done` flag breaks out of the FILE loop immediately after
    max_total_rows is hit — the next file is never opened.
    """

    def __init__(self, dataset_path: Path, cfg: dict):
        self.dataset_path = Path(dataset_path)
        self.cfg          = cfg

        self.files = sorted(
            glob(str(self.dataset_path / "**" / "*.parquet"), recursive=True)
        )
        if not self.files:
            raise FileNotFoundError(
                f"No parquet files found under {self.dataset_path}"
            )
        print(f"[MerRecSessionStreamer] {len(self.files)} parquet files found")

    # ------------------------------------------------------------------
    def _iter_files(self) -> list:
        return self.files[:]

    # ------------------------------------------------------------------
    @staticmethod
    def _new_state(ts: float, item: str) -> dict:
        return {
            "session_start": ts,
            "last_ts":       ts,
            "n_events":      1,
            "item_counts":   {item: 1},
            "iets":          [],
        }

    # ------------------------------------------------------------------
    @staticmethod
    def _emit(user_id: str, state: dict) -> dict:
        n_events    = state["n_events"]
        item_counts = state["item_counts"]
        n_unique    = len(item_counts)
        duration    = state["last_ts"] - state["session_start"]

        revisit_rate = (
            sum(1 for c in item_counts.values() if c > 1) / n_unique
            if n_unique > 0 else 0.0
        )
        iets       = state["iets"]
        iet_median = float(np.median(iets)) if iets else 0.0
        velocity   = n_events / max(duration, 1.0)

        return {
            "user_id":                 user_id,
            "session_length_s":        float(duration),
            "n_events":                n_events,
            "n_unique_items":          n_unique,
            "revisit_rate":            revisit_rate,
            "inter_event_time_median": iet_median,
            "session_velocity":        float(velocity),
        }

    # ------------------------------------------------------------------
    def __iter__(self):
        col_u      = self.cfg["col_user"]
        col_i      = self.cfg["col_item"]
        col_a      = self.cfg["col_action"]
        col_t      = self.cfg["col_time"]
        pos_action = self.cfg["positive_action"]
        gap_s      = self.cfg["session_gap_s"]
        min_events = self.cfg["min_session_events"]
        max_total  = self.cfg["max_total_rows"]

        user_state = {}
        total_rows = 0
        done       = False          # ← stops next file being opened

        for fp in self._iter_files():

            if done:                # ← checked BEFORE ds.dataset() is called
                break

            try:
                dataset      = ds.dataset(fp, format="parquet")
                schema_names = dataset.schema.names

                item_col = col_i if col_i in schema_names else "product_id"
                if item_col not in schema_names:
                    print(f"[WARN] no item column in {fp}, skipping")
                    continue
                if col_t not in schema_names:
                    print(f"[WARN] no timestamp column in {fp}, skipping")
                    continue

                scanner = dataset.scanner(
                    columns=[col_u, item_col, col_a, col_t],
                    batch_size=65536,
                )

                for record_batch in scanner.to_batches():

                    users   = record_batch.column(col_u).to_pylist()
                    items   = record_batch.column(item_col).to_pylist()
                    actions = record_batch.column(col_a).to_pylist()
                    times   = record_batch.column(col_t).to_pylist()

                    for u, it, act, ts_raw in zip(users, items, actions, times):

                        if total_rows >= max_total:
                            for uid, st in user_state.items():
                                if st["n_events"] >= min_events:
                                    yield self._emit(uid, st)
                            done = True     # ← set flag
                            break           # ← break row loop

                        if act != pos_action:
                            continue
                        if u is None or it is None or ts_raw is None:
                            continue

                        try:
                            ts = float(ts_raw)
                        except (TypeError, ValueError):
                            continue

                        total_rows += 1
                        it_str = str(it)

                        if u not in user_state:
                            user_state[u] = self._new_state(ts, it_str)
                        else:
                            st  = user_state[u]
                            gap = ts - st["last_ts"]

                            if gap > gap_s:
                                if st["n_events"] >= min_events:
                                    yield self._emit(u, st)
                                user_state[u] = self._new_state(ts, it_str)
                            else:
                                st["iets"].append(gap)
                                st["last_ts"]  = ts
                                st["n_events"] += 1
                                st["item_counts"][it_str] = (
                                    st["item_counts"].get(it_str, 0) + 1
                                )

                    if done:        # ← break batch loop
                        break

            except Exception as e:
                print(f"[WARN] Failed reading {fp}: {e}")
                continue

        # Final flush — only reached if we exhausted all files naturally
        if not done:
            for uid, st in user_state.items():
                if st["n_events"] >= min_events:
                    yield self._emit(uid, st)


print("MerRecSessionStreamer defined.")

MerRecSessionStreamer defined.


In [13]:
import pyarrow as pa
import pyarrow.parquet as pq

OUTPUT_PATH = Path(CFG["output_dir"]) / "sessions.parquet"

def stream_sessions_to_parquet(streamer, out_path, chunk_size=20_000):
    buffer = []
    writer = None

    for session in tqdm(streamer, desc="Streaming → Parquet"):
        buffer.append(session)

        if len(buffer) >= chunk_size:
            df = pd.DataFrame(buffer)
            table = pa.Table.from_pandas(df)

            if writer is None:
                writer = pq.ParquetWriter(out_path, table.schema)

            writer.write_table(table)
            buffer = []

    if buffer:
        df = pd.DataFrame(buffer)
        table = pa.Table.from_pandas(df)
        writer.write_table(table)

    if writer:
        writer.close()

    print(f"Saved → {out_path}")

In [ ]:
streamer = MerRecSessionStreamer(DATASET_PATH, CFG)
stream_sessions_to_parquet(streamer, OUTPUT_PATH)

[MerRecSessionStreamer] 2170 parquet files found


Streaming → Parquet: 0it [00:00, ?it/s]

In [ ]:
def sample_sessions(dataset_path, sample_size=100_000):
    dataset = ds.dataset(dataset_path, format="parquet")
    scanner = dataset.scanner(batch_size=100_000)

    collected = []
    total = 0

    for batch in scanner.to_batches():
        df = batch.to_pandas()
        collected.append(df)
        total += len(df)

        if total >= sample_size * 2:
            break

    df_all = pd.concat(collected, ignore_index=True)

    return df_all.sample(sample_size, random_state=RANDOM_STATE)


df_sessions = sample_sessions(OUTPUT_PATH, 100_000)

print("Sample:", df_sessions.shape)

In [ ]:
def build_feature_matrix(df, cfg):
    features = cfg["features"]
    X = df[features].copy().astype(float)

    # Clip
    for col in features:
        X[col] = X[col].clip(upper=X[col].quantile(0.995))

    # Fit imputer
    imputer = SimpleImputer(strategy="median")
    X = pd.DataFrame(imputer.fit_transform(X), columns=features)

    # Log
    for col in cfg["log_transform_cols"]:
        if col in features:
            X[col] = np.log1p(X[col])

    # Scale
    Scaler = RobustScaler if cfg["scaler"] == "robust" else StandardScaler
    scaler = Scaler()

    X_scaled = scaler.fit_transform(X)

    return X_scaled, scaler, imputer


X, scaler, imputer = build_feature_matrix(df_sessions, CFG)

joblib.dump(scaler,  "gmm_scaler.pkl")
joblib.dump(imputer, "gmm_imputer.pkl")

In [ ]:
def gmm_model_selection(X, cfg):
    results = []

    for cov in cfg["covariance_types"]:
        for k in cfg["k_range"]:

            gmm = GaussianMixture(
                n_components=k,
                covariance_type=cov,
                n_init=cfg["n_init"],
                max_iter=cfg["max_iter"],
                random_state=RANDOM_STATE,
            )

            gmm.fit(X)
            labels = gmm.predict(X)

            sil = silhouette_score(X, labels)

            results.append({
                "k": k,
                "cov": cov,
                "bic": gmm.bic(X),
                "sil": sil,
            })

    return pd.DataFrame(results).sort_values("bic")


selection_df = gmm_model_selection(X, CFG)
BEST_K = int(selection_df.iloc[0]["k"])

In [ ]:
final_gmm = GaussianMixture(
    n_components=BEST_K,
    covariance_type=CFG["final_cov_type"],
    n_init=5,
    max_iter=CFG["max_iter"],
    random_state=RANDOM_STATE,
)

final_gmm.fit(X)

joblib.dump(final_gmm, "gmm_model.pkl")

In [ ]:
def stream_inference(dataset_path, out_path, gmm, scaler, imputer, cfg):
    dataset = ds.dataset(dataset_path, format="parquet")
    scanner = dataset.scanner(batch_size=50_000)

    writer = None

    for batch in tqdm(scanner.to_batches(), desc="Inference"):
        df = batch.to_pandas()

        X = df[cfg["features"]].copy().astype(float)

        # SAME preprocessing
        for col in cfg["features"]:
            X[col] = X[col].clip(upper=X[col].quantile(0.995))

        X = pd.DataFrame(imputer.transform(X), columns=cfg["features"])

        for col in cfg["log_transform_cols"]:
            if col in cfg["features"]:
                X[col] = np.log1p(X[col])

        X_scaled = scaler.transform(X)

        labels = gmm.predict(X_scaled)
        probs  = gmm.predict_proba(X_scaled)

        df["gmm_regime"] = labels
        df["confidence"] = probs.max(axis=1)

        table = pa.Table.from_pandas(df)

        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema)

        writer.write_table(table)

    if writer:
        writer.close()

    print(f"Saved full labeled dataset → {out_path}")